In [ ]:
from pathlib import Path

import soundfile as sf
from transformers import SpeechT5ForSpeechToSpeech, SpeechT5Processor

# Define the path to your audio file
audio_file_path = Path("../data/speechocean762/train/audios/000002.wav")

audio_array, sampling_rate = sf.read(str(audio_file_path))

processor = SpeechT5Processor.from_pretrained("microsoft/speecht5_vc")
model = SpeechT5ForSpeechToSpeech.from_pretrained("microsoft/speecht5_vc")

inputs = processor(audio=audio_array, sampling_rate=sampling_rate, return_tensors="pt")


/Users/masaishi/ghq/github.com/masaishi/claion-exp/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Some weights of SpeechT5ForSpeechToSpeech were not initialized from the model checkpoint at microsoft/speecht5_vc and are newly initialized: ['speecht5.encoder.prenet.pos_sinusoidal_embed.weights']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [3]:
from transformers.models.speecht5.modeling_speecht5 import SpeechT5EncoderWithSpeechPrenet

isinstance(model.speecht5.encoder, SpeechT5EncoderWithSpeechPrenet)

True

In [ ]:
from pathlib import Path

import soundfile as sf
import torch
from transformers import SpeechT5ForSpeechToSpeech, SpeechT5Processor

# Define the path to your audio file
audio_file_path = Path("../data/speechocean762/train/audios/000002.wav")
audio_array, sampling_rate = sf.read(str(audio_file_path))

processor = SpeechT5Processor.from_pretrained("microsoft/speecht5_vc")
model = SpeechT5ForSpeechToSpeech.from_pretrained("microsoft/speecht5_vc")

inputs = processor(audio=audio_array, sampling_rate=sampling_rate, return_tensors="pt")

speaker_embeddings = torch.randn(1, 512)  # Speaker embedding for voice conversion
threshold = 0.5
minlenratio = 0.0
maxlenratio = 20.0
vocoder = None
output_cross_attentions = False
return_output_lengths = False

# Process the audio
with torch.no_grad():
    input_values = inputs.input_values
    attention_mask = inputs.attention_mask

    if attention_mask is None:
        encoder_attention_mask = 1 - (input_values == model.config.pad_token_id).int()
    else:
        encoder_attention_mask = attention_mask

    bsz = input_values.size(0)

    encoder_out = model.speecht5.encoder(
        input_values=input_values,
        attention_mask=encoder_attention_mask,
        return_dict=True,
    )

    encoder_last_hidden_state = encoder_out.last_hidden_state

    encoder_attention_mask = model.speecht5.encoder.prenet._get_feature_vector_attention_mask(encoder_out[0].shape[1], encoder_attention_mask)

    maxlen = int(encoder_last_hidden_state.size(1) * maxlenratio / model.config.reduction_factor)
    minlen = int(encoder_last_hidden_state.size(1) * minlenratio / model.config.reduction_factor)

    # Start the output sequence with a mel spectrum that is all zeros.
    output_sequence = encoder_last_hidden_state.new_zeros(bsz, 1, model.config.num_mel_bins)

    spectrogram = []
    cross_attentions = []
    past_key_values = None
    idx = 0
    result_spectrogram = {}

    while True:
        idx += 1

        # Run the decoder prenet on the entire output sequence.
        decoder_hidden_states = model.speecht5.decoder.prenet(output_sequence, speaker_embeddings)
        # Run the decoder layers on the last element of the prenet output.
        decoder_out = model.speecht5.decoder.wrapped_decoder(
            hidden_states=decoder_hidden_states[:, -1:],
            attention_mask=None,
            encoder_hidden_states=encoder_last_hidden_state,
            encoder_attention_mask=encoder_attention_mask,
            past_key_values=past_key_values,
            use_cache=True,
            output_attentions=output_cross_attentions,
            return_dict=True,
        )

        if output_cross_attentions:
            cross_attentions.append(torch.cat(decoder_out.cross_attentions, dim=0))

        last_decoder_output = decoder_out.last_hidden_state.squeeze(1)
        past_key_values = decoder_out.past_key_values

        # Predict the new mel spectrum for this step in the sequence.
        spectrum = model.speech_decoder_postnet.feat_out(last_decoder_output)
        spectrum = spectrum.view(bsz, model.config.reduction_factor, model.config.num_mel_bins)
        spectrogram.append(spectrum)

        # Extend the output sequence with the new mel spectrum.
        new_spectrogram = spectrum[:, -1, :].view(bsz, 1, model.config.num_mel_bins)
        output_sequence = torch.cat((output_sequence, new_spectrogram), dim=1)
        # Predict the probability that this is the stop token.
        prob = torch.sigmoid(model.speech_decoder_postnet.prob_out(last_decoder_output))

        if idx < minlen:
            continue
        else:
            # If the generation loop is less than maximum length time, check the ones in the batch that have met
            # the prob threshold. Otherwise, assume all have met thresholds and fill other spectrograms for the batch.
            if idx < maxlen:
                meet_thresholds = torch.sum(prob, dim=-1) >= threshold
                meet_indexes = torch.where(meet_thresholds)[0].tolist()
            else:
                meet_indexes = range(len(prob))
            meet_indexes = [i for i in meet_indexes if i not in result_spectrogram]
            if len(meet_indexes) > 0:
                spectrograms = torch.stack(spectrogram)
                spectrograms = spectrograms.transpose(0, 1).flatten(1, 2)
                spectrograms = model.speech_decoder_postnet.postnet(spectrograms)
                for meet_index in meet_indexes:
                    result_spectrogram[meet_index] = spectrograms[meet_index]
            if len(result_spectrogram) >= bsz:
                break
    spectrograms = [result_spectrogram[i] for i in range(len(result_spectrogram))]
    if not return_output_lengths:
        spectrogram = spectrograms[0] if bsz == 1 else torch.nn.utils.rnn.pad_sequence(spectrograms, batch_first=True)
        if vocoder is not None:
            outputs = vocoder(spectrogram)
        else:
            outputs = spectrogram
        if output_cross_attentions:
            cross_attentions = torch.cat(cross_attentions, dim=2)
            if bsz > 1:
                cross_attentions = cross_attentions.view(bsz, int(cross_attentions.size(0) / bsz), *cross_attentions.size()[-3:])
            outputs = (outputs, cross_attentions)
    else:
        # batched return values should also include the spectrogram/waveform lengths
        spectrogram_lengths = []
        for i in range(bsz):
            spectrogram_lengths.append(spectrograms[i].size(0))
        if vocoder is None:
            spectrograms = torch.nn.utils.rnn.pad_sequence(spectrograms, batch_first=True)
            outputs = (spectrograms, spectrogram_lengths)
        else:
            waveforms = []
            spectrograms = torch.nn.utils.rnn.pad_sequence(spectrograms, batch_first=True)
            waveforms = vocoder(spectrograms)
            waveform_lengths = [int(waveforms.size(1) / max(spectrogram_lengths)) * i for i in spectrogram_lengths]
            outputs = (waveforms, waveform_lengths)
        if output_cross_attentions:
            cross_attentions = torch.cat(cross_attentions, dim=2)
            cross_attentions = cross_attentions.view(bsz, int(cross_attentions.size(0) / bsz), *cross_attentions.size()[-3:])
            outputs = (*outputs, cross_attentions)

outputs, outputs.shape


Some weights of SpeechT5ForSpeechToSpeech were not initialized from the model checkpoint at microsoft/speecht5_vc and are newly initialized: ['speecht5.encoder.prenet.pos_sinusoidal_embed.weights']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


(tensor([[-4.3407, -4.4120, -4.3949,  ..., -4.2544, -4.2257, -4.0780],
         [-4.0191, -4.1701, -4.2904,  ..., -4.2035, -4.1554, -4.1430],
         [-3.6834, -3.9188, -4.0913,  ..., -4.2860, -4.2893, -4.3917],
         ...,
         [-3.5650, -3.6675, -3.7819,  ..., -4.4175, -4.4016, -4.4812],
         [-3.8415, -3.9734, -4.0656,  ..., -4.2057, -4.1656, -4.2759],
         [-4.1122, -4.2472, -4.2659,  ..., -4.1194, -4.1032, -4.2120]]),
 torch.Size([118, 80]))

In [ ]:
import numpy as np
import soundfile as sf
import torch

# Your tensor data (already loaded as 'outputs' in your code)
# shape: [118, 80]
spectrogram_db = outputs.detach().cpu().numpy()

# Convert from dB to linear scale
spectrogram_mag = 10 ** (spectrogram_db / 10)

# Parameters
sample_rate = 22050
n_mels = 80  # Your frequency dimension
hop_length = 256

# Method 1: Use torchaudio for mel inversion if available
try:
    import torchaudio

    # Create MelScale and InverseMelScale
    mel_scale = torchaudio.transforms.InverseMelScale(n_stft=(n_mels - 1) * 2, n_mels=n_mels, sample_rate=sample_rate)

    # Convert to torch tensor
    mel_spec_tensor = torch.tensor(spectrogram_mag).float()

    # Inverse mel transform
    linear_spec = mel_scale(mel_spec_tensor)

    # Griffin-Lim on the linear spectrogram
    reconstructed_audio = torchaudio.transforms.GriffinLim(n_fft=(n_mels - 1) * 2, hop_length=hop_length, win_length=(n_mels - 1) * 2)(linear_spec)

    # Convert to numpy array
    reconstructed_audio = reconstructed_audio.squeeze().numpy()

except ImportError:
    # Method 2: Use direct signal processing approach
    # Create time-domain signal with the right shape
    time_frames = spectrogram_mag.shape[0]
    audio_length = hop_length * (time_frames - 1) + hop_length

    # Simple approach: create a time domain signal with random phase
    reconstructed_audio = np.zeros(audio_length)

    # For each time frame
    for i in range(time_frames):
        # Create a synthetic sinusoidal component for each frequency bin
        for j in range(n_mels):
            # Use the magnitude from the spectrogram and a random phase
            mag = spectrogram_mag[i, j]
            if mag > 0:  # Only add if magnitude is positive
                # Synthesize frequency component (crude approximation)
                freq = (j + 1) * (sample_rate / (2 * n_mels))
                t = np.arange(hop_length) / sample_rate
                # Add to the signal at the right position
                start_idx = i * hop_length
                reconstructed_audio[start_idx : start_idx + hop_length] += mag * np.sin(2 * np.pi * freq * t)

    # Normalize
    reconstructed_audio = reconstructed_audio / np.max(np.abs(reconstructed_audio))

# Save as WAV file
sf.write("reconstructed_audio.wav", reconstructed_audio, sample_rate)
print("Reconstruction complete! Audio saved as 'reconstructed_audio.wav'")


ValueError: could not broadcast input array from shape (80,80) into shape (118,80)